In [3]:
import pandas as pd
import numpy as np

RANDOM_SEED = 42

# REMEMBER: use only train.csv in this whole phase, never test.csv
train_df = pd.read_csv("../data/processed/train.csv")

feature_cols = [f"f{i}" for i in range(63)]
X_train = train_df[feature_cols].to_numpy(dtype=np.float64)
y_train = train_df["label"].astype(str).to_numpy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("Classes:", len(np.unique(y_train)))

X_train shape: (2165, 63)
y_train shape: (2165,)
Classes: 36


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

baseline = DummyClassifier(strategy="most_frequent")

scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="accuracy")

print("Scores of the 5 folds:", np.round(scores, 4))
print("Average accuracy:", round(scores.mean(), 4))
print("Most frequent (majority) class:", pd.Series(y_train).value_counts().idxmax())

Scores of the 5 folds: [0.03   0.03   0.03   0.0323 0.03  ]
Average accuracy: 0.0305
Most frequent (majority) class: Bay


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_scores = cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring="accuracy")

print("KNN scores of the 5 folds:", np.round(knn_scores, 4))
print("KNN average accuracy:", round(knn_scores.mean(), 4))

KNN scores of the 5 folds: [0.806  0.8406 0.8522 0.8776 0.7483]
KNN average accuracy: 0.8249


In [6]:
from sklearn.svm import SVC

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf"))
])

svm_scores = cross_val_score(svm_pipeline, X_train, y_train, cv=5, scoring="accuracy")

print("SVM scores of the 5 folds:", np.round(svm_scores, 4))
print("SVM average accuracy:", round(svm_scores.mean(), 4))

SVM scores of the 5 folds: [0.843  0.8799 0.8822 0.9261 0.8152]
SVM average accuracy: 0.8693


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED))
])

rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring="accuracy")

print("Random Forest scores of the 5 folds:", np.round(rf_scores, 4))
print("Random Forest average accuracy:", round(rf_scores.mean(), 4))

Random Forest scores of the 5 folds: [0.8707 0.8891 0.8938 0.9261 0.8453]
Random Forest average accuracy: 0.885


In [8]:
from sklearn.neural_network import MLPClassifier

mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=RANDOM_SEED))
])

mlp_scores = cross_val_score(mlp_pipeline, X_train, y_train, cv=5, scoring="accuracy")

print("MLP scores of the 5 folds:", np.round(mlp_scores, 4))
print("MLP average accuracy:", round(mlp_scores.mean(), 4))

MLP scores of the 5 folds: [0.9145 0.9169 0.9215 0.9376 0.8776]
MLP average accuracy: 0.9136


In [11]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)

In [12]:
results_summary = pd.DataFrame({
    "Model": ["Baseline (majority class)", "KNN", "SVM (RBF)", "Random Forest", "MLP"],
    "Fold 1": [scores[0], knn_scores[0], svm_scores[0], rf_scores[0], mlp_scores[0]],
    "Fold 2": [scores[1], knn_scores[1], svm_scores[1], rf_scores[1], mlp_scores[1]],
    "Fold 3": [scores[2], knn_scores[2], svm_scores[2], rf_scores[2], mlp_scores[2]],
    "Fold 4": [scores[3], knn_scores[3], svm_scores[3], rf_scores[3], mlp_scores[3]],
    "Fold 5": [scores[4], knn_scores[4], svm_scores[4], rf_scores[4], mlp_scores[4]],
})

results_summary["Average"] = results_summary[["Fold 1","Fold 2","Fold 3","Fold 4","Fold 5"]].mean(axis=1)
results_summary["Std Dev"] = results_summary[["Fold 1","Fold 2","Fold 3","Fold 4","Fold 5"]].std(axis=1)

results_summary = results_summary.round(4)
results_summary.to_csv("../results/cv_results.csv", index=False)

results_summary

,Model,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Average,Std Dev
0,Baseline (majority class),0.0300,0.0300,0.0300,0.0323,0.0300,0.0305,0.0010
1,KNN,0.8060,0.8406,0.8522,0.8776,0.7483,0.8249,0.0500
2,SVM (RBF),0.8430,0.8799,0.8822,0.9261,0.8152,0.8693,0.0422
3,Random Forest,0.8707,0.8891,0.8938,0.9261,0.8453,0.8850,0.0299
4,MLP,0.9145,0.9169,0.9215,0.9376,0.8776,0.9136,0.0221


In [13]:
from sklearn.model_selection import GridSearchCV

knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid=knn_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

print("Best K:", knn_grid.best_params_)
print("Best CV accuracy:", round(knn_grid.best_score_, 4))

Best K: {'model__n_neighbors': 3}
Best CV accuracy: 0.8277


In [14]:
svm_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1]
}

svm_grid = GridSearchCV(
    svm_pipeline,
    param_grid=svm_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print("Best params:", svm_grid.best_params_)
print("Best CV accuracy:", round(svm_grid.best_score_, 4))

Best params: {'model__C': 100, 'model__gamma': 0.01}
Best CV accuracy: 0.9219


In [15]:
rf_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best params:", rf_grid.best_params_)
print("Best CV accuracy:", round(rf_grid.best_score_, 4))

Best params: {'model__max_depth': 20, 'model__n_estimators': 300}
Best CV accuracy: 0.8868


In [16]:
mlp_param_grid = {
    "model__hidden_layer_sizes": [(50,), (100,), (100, 50)],
    "model__learning_rate_init": [0.001, 0.01]
}

mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=mlp_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

mlp_grid.fit(X_train, y_train)

print("Best params:", mlp_grid.best_params_)
print("Best CV accuracy:", round(mlp_grid.best_score_, 4))

Best params: {'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.001}
Best CV accuracy: 0.9201


In [17]:
tuning_summary = pd.DataFrame({
    "Model": ["KNN", "SVM (RBF)", "Random Forest", "MLP"],
    "Default CV Accuracy": [
        round(knn_scores.mean(), 4),
        round(svm_scores.mean(), 4),
        round(rf_scores.mean(), 4),
        round(mlp_scores.mean(), 4),
    ],
    "Tuned CV Accuracy": [
        round(knn_grid.best_score_, 4),
        round(svm_grid.best_score_, 4),
        round(rf_grid.best_score_, 4),
        round(mlp_grid.best_score_, 4),
    ],
    "Best Hyperparameters": [
        str(knn_grid.best_params_),
        str(svm_grid.best_params_),
        str(rf_grid.best_params_),
        str(mlp_grid.best_params_),
    ],
})

tuning_summary["Improvement"] = tuning_summary["Tuned CV Accuracy"] - tuning_summary["Default CV Accuracy"]
tuning_summary = tuning_summary.sort_values("Tuned CV Accuracy", ascending=False)

tuning_summary.to_csv("../results/tuning_results.csv", index=False)
tuning_summary

,Model,Default CV Accuracy,Tuned CV Accuracy,Best Hyperparameters,Improvement
1,SVM (RBF),0.8693,0.9219,"{'model__C': 100, 'model__gamma': 0.01}",0.0526
3,MLP,0.9136,0.9201,"{'model__hidden_layer_sizes': (100, 50), 'mode...",0.0065
2,Random Forest,0.8850,0.8868,"{'model__max_depth': 20, 'model__n_estimators'...",0.0018
0,KNN,0.8249,0.8277,{'model__n_neighbors': 3},0.0028
